# PCA-SPC Industrial Monitoring — Generic Template v2

**Author:** Daniele Marangon | **GitHub:** [MarDan93](https://github.com/MarDan93)

---

## Workflow
```
Raw Data → Exploratory Analysis (trends + correlations X+Y)
         → Split: temporal / random / stratified
         → Iterative Phase I Cleaning (removes contamination)
         → PCA Component Selection (Scree + Kaiser + RMSECV)
         → Phase I Model Fit → Phase II Monitoring (T² + Q)
         → Contribution Plots → Y Validation → Final Report
```

## Split guide
| Dataset type | `SPLIT_TYPE` |
|---|---|
| Continuous process — press, extruder, reactor | `temporal` |
| Independent batches, lab data, public datasets | `random` |
| Imbalanced fault detection | `stratified` |

## How to use on a new dataset
**Edit only Section 1.** Everything else adapts automatically.


---
# SECTION 1 — CONFIGURATION
> ⚠️ **Edit only this section to adapt the template to a new dataset.**

In [ ]:
# ============================================================
#  CONFIGURATION
# ============================================================

# --- 1) Data source: 'url', 'drive', 'local' ---
DATA_SOURCE      = 'url'
DATA_URL         = 'https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data'
DATA_DRIVE_PATH  = '/content/drive/My Drive/Colab_Notebooks/Dataset/your_file.xlsx'
DATA_LOCAL_PATH  = './data/your_file.csv'

# --- 2) File format: 'csv' or 'excel' ---
FILE_FORMAT  = 'csv'
CSV_SEP      = ' '   # SECOM = space-separated; standard CSV = ','
EXCEL_HEADER = 0

# --- 3) Column roles ---
# X_COLS: process variables for PCA. Leave [] to auto-detect.
# Y_COLS: quality/output variables — excluded from PCA, used for validation.
# COLUMNS_TO_EXCLUDE: IDs, timestamps, index columns to drop.
#
# Example — injection molding:
#   Y_COLS = ['Quota cuscino', 'Tempo ciclo', 'Quota minimo cuscino']
#   COLUMNS_TO_EXCLUDE = ['1']
X_COLS             = []
Y_COLS             = []
COLUMNS_TO_EXCLUDE = []

# --- 4) Split strategy ---
# 'temporal'  : first TRAIN_SIZE rows = Phase I  → continuous processes
# 'random'    : random 70/30 shuffle             → independent observations
# 'stratified': preserves class proportions      → imbalanced fault data
SPLIT_TYPE   = 'random'   # change to 'temporal' for injection molding
TRAIN_SIZE   = 0.70
RANDOM_STATE = 42
STRATIFY_COL = None       # only used when SPLIT_TYPE = 'stratified'

# --- 5) Iterative Phase I cleaning ---
# Removes contaminated (anomalous) cycles from training set.
# Recommended = True for temporal datasets with unknown stability.
# Set False only if training data is certified in-control.
PHASE1_CLEANING    = True
PHASE1_MAX_ITER    = 10
PHASE1_ALPHA_CLEAN = 0.99  # stricter than monitoring — removes only clear outliers

# --- 6) PCA-SPC parameters ---
N_COMPONENTS        = None  # None = automatic via RMSECV
ALPHA_CONTROL       = 0.95
MAX_COMPONENTS_EVAL = 30
CV_FOLDS            = 10

# --- 7) Tolerance limits for Y variables ---
# Format:
#   'col': ('absolute', delta)       -> mean ± delta
#   'col': ('relative', fraction)    -> mean ± fraction*mean
#   'col': ('fixed', lower, upper)   -> fixed spec limits
# Leave {} for automatic ±2sigma.
#
# Example — injection molding:
#   TOLERANCE_SPECS = {
#       'Quota cuscino':        ('absolute', 0.5),
#       'Quota minimo cuscino': ('absolute', 0.5),
#       'Tempo ciclo':          ('relative', 0.02),
#   }
TOLERANCE_SPECS = {}

# --- 8) Contribution plots ---
TOP_N_CONTRIB      = 5
USE_VAR_NAMES_ON_X = False

print('✅ Configuration loaded.')
print(f'   Split type       : {SPLIT_TYPE}')
print(f'   Phase I cleaning : {PHASE1_CLEANING}')
print(f'   Y variables      : {Y_COLS if Y_COLS else "none defined"}')

---
# SECTION 2 — IMPORTS & DATA LOADING

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import f, norm
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
print('✅ Imports OK.')

In [ ]:
def load_data(source, url, drive_path, local_path, file_format, sep, header):
    if source == 'drive':
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        path = drive_path
    elif source == 'url':   path = url
    elif source == 'local': path = local_path
    else: raise ValueError(f'Unknown DATA_SOURCE: {source}')
    if file_format == 'csv':
        df = pd.read_csv(path, sep=sep, header=None if source=='url' else header)
    elif file_format == 'excel':
        df = pd.read_excel(path, header=header)
    else: raise ValueError(f'Unknown FILE_FORMAT: {file_format}')
    return df

df_raw = load_data(DATA_SOURCE, DATA_URL, DATA_DRIVE_PATH, DATA_LOCAL_PATH,
                   FILE_FORMAT, CSV_SEP, EXCEL_HEADER)
df_raw.columns = df_raw.columns.astype(str)
print(f'✅ Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head()

---
# SECTION 3 — PREPROCESSING

In [ ]:
df_work = df_raw.copy()

# Drop excluded columns
cols_drop = [c for c in COLUMNS_TO_EXCLUDE if c in df_work.columns]
if cols_drop: df_work.drop(columns=cols_drop, inplace=True)

# Keep numeric only
df_work = df_work.select_dtypes(include=[np.number])

# Handle missing values: drop >50% missing cols, impute rest with mean
missing = df_work.isnull().sum()
high_miss = missing[missing > 0.5 * len(df_work)].index.tolist()
if high_miss:
    df_work.drop(columns=high_miss, inplace=True)
    print(f'Dropped {len(high_miss)} columns with >50% missing')
df_work.fillna(df_work.mean(), inplace=True)

# Drop zero-variance columns
const_cols = df_work.columns[df_work.std() == 0].tolist()
if const_cols:
    df_work.drop(columns=const_cols, inplace=True)
    print(f'Dropped {len(const_cols)} constant columns')

# Separate X and Y
Y_COLS_valid = [c for c in Y_COLS if c in df_work.columns]
X_COLS_valid = [c for c in X_COLS if c in df_work.columns] if X_COLS \
               else [c for c in df_work.columns if c not in Y_COLS_valid]
FEATURE_NAMES = X_COLS_valid

df_X = df_work[X_COLS_valid].reset_index(drop=True)
df_Y = df_work[Y_COLS_valid].reset_index(drop=True) if Y_COLS_valid else pd.DataFrame()

print(f'\n✅ Preprocessing complete.')
print(f'   X (process variables) : {len(X_COLS_valid)}')
print(f'   Y (quality variables) : {len(Y_COLS_valid)} → {Y_COLS_valid}')
print(f'   Observations          : {len(df_X)}')

---
# SECTION 4 — EXPLORATORY ANALYSIS

> Before splitting, explore the full dataset (X + Y together):
> 1. **Temporal trends** — is the process stable? where to split?
> 2. **Correlation heatmap** — which X variables correlate with Y? (justifies future PLS)

In [ ]:
# --- Temporal trend plots (first 12 X variables) ---
vars_to_plot = FEATURE_NAMES[:12]
ncols = 3; nrows = int(np.ceil(len(vars_to_plot)/ncols))
split_line = int(len(df_X)*TRAIN_SIZE) if SPLIT_TYPE=='temporal' else None

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows*2.5))
axes = axes.flatten()
for i, col in enumerate(vars_to_plot):
    axes[i].plot(df_X[col].values, lw=0.8, alpha=0.8)
    if split_line:
        axes[i].axvline(split_line, color='red', linestyle='--', lw=1.2, label='Train|Test')
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Observation'); axes[i].grid(True, alpha=0.3)
for j in range(len(vars_to_plot), len(axes)): axes[j].set_visible(False)
if split_line: axes[0].legend(fontsize=8)
plt.suptitle('Temporal Trends — X Variables (first 12)', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

if split_line:
    print(f'Red line = train/test boundary at observation {split_line}')
print('Look for: trends, jumps, increasing variance — especially in the train region.')

In [ ]:
# --- Correlation heatmap: X + Y together ---
df_full = pd.concat([df_X, df_Y], axis=1) if not df_Y.empty else df_X.copy()
corr    = df_full.corr()

# Cap at 30 variables for readability
if len(corr) > 30:
    print('Showing first 30 variables for readability.')
    corr = corr.iloc[:30, :30]

fig_sz = max(10, len(corr)*0.45)
plt.figure(figsize=(fig_sz, fig_sz*0.85))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=(len(corr)<=20), fmt='.2f', linewidths=0.3,
            cbar_kws={'label': 'Pearson r'})

# Highlight Y labels in red
ax = plt.gca()
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    if tick.get_text() in Y_COLS_valid:
        tick.set_color('red'); tick.set_fontweight('bold')

plt.title('Correlation Heatmap — X + Y (Y in red)' if Y_COLS_valid else 'Correlation Heatmap')
plt.tight_layout(); plt.show()

# Print strongest X→Y correlations
if Y_COLS_valid:
    print('\nStrongest X→Y correlations (|r| > 0.3):')
    for y_col in Y_COLS_valid:
        if y_col not in corr.columns: continue
        corr_y = corr[y_col].drop(labels=Y_COLS_valid, errors='ignore')
        strong = corr_y[corr_y.abs()>0.3].sort_values(key=abs, ascending=False)
        print(f'\n  {y_col}:')
        if len(strong)>0:
            for x_var, r in strong.items(): print(f'    {x_var:40s}  r={r:+.3f}')
        else:
            print('    No strong linear correlations found.')
            print('    Note: nonlinear relations may still exist — PLS will test this.')

---
# SECTION 5 — TRAIN / TEST SPLIT

In [ ]:
n_total = len(df_X)

if SPLIT_TYPE == 'temporal':
    # Preserves time order — correct for continuous processes
    n_train = int(n_total * TRAIN_SIZE)
    df_train_X = df_X.iloc[:n_train].reset_index(drop=True)
    df_test_X  = df_X.iloc[n_train:].reset_index(drop=True)
    df_train_Y = df_Y.iloc[:n_train].reset_index(drop=True) if not df_Y.empty else pd.DataFrame()
    df_test_Y  = df_Y.iloc[n_train:].reset_index(drop=True) if not df_Y.empty else pd.DataFrame()

elif SPLIT_TYPE == 'random':
    idx_tr, idx_te = train_test_split(np.arange(n_total), train_size=TRAIN_SIZE,
                                      random_state=RANDOM_STATE, shuffle=True)
    df_train_X = df_X.iloc[idx_tr].reset_index(drop=True)
    df_test_X  = df_X.iloc[idx_te].reset_index(drop=True)
    df_train_Y = df_Y.iloc[idx_tr].reset_index(drop=True) if not df_Y.empty else pd.DataFrame()
    df_test_Y  = df_Y.iloc[idx_te].reset_index(drop=True) if not df_Y.empty else pd.DataFrame()

elif SPLIT_TYPE == 'stratified':
    if not STRATIFY_COL or STRATIFY_COL not in df_work.columns:
        raise ValueError('SPLIT_TYPE=stratified requires a valid STRATIFY_COL.')
    idx_tr, idx_te = train_test_split(np.arange(n_total), train_size=TRAIN_SIZE,
                                      random_state=RANDOM_STATE,
                                      stratify=df_work[STRATIFY_COL].values)
    df_train_X = df_X.iloc[idx_tr].reset_index(drop=True)
    df_test_X  = df_X.iloc[idx_te].reset_index(drop=True)
    df_train_Y = df_Y.iloc[idx_tr].reset_index(drop=True) if not df_Y.empty else pd.DataFrame()
    df_test_Y  = df_Y.iloc[idx_te].reset_index(drop=True) if not df_Y.empty else pd.DataFrame()
else:
    raise ValueError(f'Unknown SPLIT_TYPE: {SPLIT_TYPE}')

print(f'✅ Split [{SPLIT_TYPE}]: Phase I = {len(df_train_X)} obs | Phase II = {len(df_test_X)} obs')

---
# SECTION 6 — ITERATIVE PHASE I CLEANING

> **Phase I contamination:** if the training set contains anomalous cycles,
> the model learns that "anomalous is normal" → control limits become too wide.
>
> **Solution:** fit, remove outliers, refit — repeat until convergence.
>
> `PHASE1_ALPHA_CLEAN = 0.99` is stricter than `ALPHA_CONTROL = 0.95`
> to remove only clear outliers, not borderline cases.

In [ ]:
def quick_t2_q(X_scaled, k, alpha, rs=42):
    """Fast T²/Q on already-scaled data. Returns outlier flags."""
    n, p = X_scaled.shape
    pca  = PCA(n_components=k, svd_solver='full', random_state=rs)
    T    = pca.fit_transform(X_scaled)
    lam  = pca.explained_variance_
    T2   = np.sum((T**2)/lam, axis=1)
    T2_UCL = (k*(n-1)/(n-k)) * f.ppf(alpha, k, n-k)
    E    = X_scaled - pca.inverse_transform(T)
    Q    = np.sum(E**2, axis=1)
    pf   = PCA(n_components=min(p,n-1), svd_solver='full', random_state=rs)
    pf.fit(X_scaled)
    re   = pf.explained_variance_[k:]
    t1,t2,t3 = re.sum(),(re**2).sum(),(re**3).sum()
    h0   = 1-(2*t1*t3)/(3*t2**2)
    z    = norm.ppf(alpha)
    Q_UCL = t1*((z*np.sqrt(2*t2*h0**2)/t1)+1+(t2*h0*(h0-1)/t1**2))**(1/h0)
    return (T2>T2_UCL)|(Q>Q_UCL), T2_UCL, Q_UCL


df_train_clean   = df_train_X.copy()
df_train_Y_clean = df_train_Y.copy() if not df_train_Y.empty else pd.DataFrame()
removed_total    = 0
cleaning_log     = []

if PHASE1_CLEANING:
    # Determine k for cleaning using Kaiser on full train
    sc_tmp = StandardScaler()
    X_tmp  = sc_tmp.fit_transform(df_train_clean[FEATURE_NAMES].values)
    pt     = PCA(svd_solver='full', random_state=RANDOM_STATE); pt.fit(X_tmp)
    k_clean = max(2, int(np.sum(pt.explained_variance_ > 1)))
    k_clean = min(k_clean, X_tmp.shape[1]-1, X_tmp.shape[0]-2)
    print(f'Phase I cleaning  |  k={k_clean} PCs (Kaiser)  |  alpha={PHASE1_ALPHA_CLEAN}')
    print()

    for it in range(1, PHASE1_MAX_ITER+1):
        n_before = len(df_train_clean)
        sc_it    = StandardScaler()
        X_it     = sc_it.fit_transform(df_train_clean[FEATURE_NAMES].values)
        k_it     = min(k_clean, X_it.shape[1]-1, X_it.shape[0]-2)
        flag_it, t2u, qu = quick_t2_q(X_it, k_it, PHASE1_ALPHA_CLEAN, RANDOM_STATE)
        n_rem    = int(flag_it.sum())
        cleaning_log.append({'Iter': it, 'Obs_before': n_before, 'Removed': n_rem,
                              'Obs_after': n_before-n_rem, 'T2_UCL': round(t2u,3), 'Q_UCL': round(qu,3)})
        print(f'  Iter {it}: {n_before} obs → removed {n_rem} → {n_before-n_rem} remaining')
        if n_rem == 0:
            print(f'\n✅ Converged at iteration {it}.')
            break
        keep = ~flag_it
        df_train_clean   = df_train_clean[keep].reset_index(drop=True)
        if not df_train_Y_clean.empty:
            df_train_Y_clean = df_train_Y_clean[keep].reset_index(drop=True)
        removed_total += n_rem
    else:
        print(f'\n⚠️  Max iterations reached without full convergence.')

    pct_removed = removed_total/len(df_train_X)*100
    print(f'\nOriginal: {len(df_train_X)} | Removed: {removed_total} ({pct_removed:.1f}%) | Clean: {len(df_train_clean)}')
    display(pd.DataFrame(cleaning_log).set_index('Iter'))

    if pct_removed > 20:
        print('\n⚠️  WARNING: >20% removed. Training period may not have been stable.')
        print('   → Review temporal trends (Section 4) and consider adjusting split boundary.')
else:
    print('Phase I cleaning skipped — assuming training data is certified stable.')

---
# SECTION 7 — NUMBER OF PRINCIPAL COMPONENTS

In [ ]:
scaler_eval    = StandardScaler()
X_train_s_eval = scaler_eval.fit_transform(df_train_clean[FEATURE_NAMES].values)
n_eval, p_eval = X_train_s_eval.shape
max_k          = min(MAX_COMPONENTS_EVAL, p_eval, n_eval-1)

pca_eval   = PCA(n_components=max_k, svd_solver='full', random_state=RANDOM_STATE)
pca_eval.fit(X_train_s_eval)
eigenvalues = pca_eval.explained_variance_
var_ratio   = pca_eval.explained_variance_ratio_
cum_var     = np.cumsum(var_ratio)
n_kaiser    = int(np.sum(eigenvalues > 1))
n_90        = int(np.argmax(cum_var >= 0.90)) + 1
n_95        = int(np.argmax(cum_var >= 0.95)) + 1
ks          = np.arange(1, max_k+1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ks, var_ratio, marker='o', linestyle='--', ms=4)
axes[0].axhline(1/p_eval, color='purple', linestyle=':', label=f'Kaiser (1/p={1/p_eval:.3f})')
axes[0].set_title('Scree Plot'); axes[0].set_xlabel('PC'); axes[0].set_ylabel('Variance ratio')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(ks, cum_var, marker='o', ms=4)
axes[1].axhline(0.90, color='orange', linestyle='--', label=f'90% → {n_90} PCs')
axes[1].axvline(n_90, color='orange', linestyle=':')
axes[1].axhline(0.95, color='green',  linestyle='--', label=f'95% → {n_95} PCs')
axes[1].axvline(n_95, color='green',  linestyle=':')
axes[1].axvline(n_kaiser, color='purple', linestyle=':', label=f'Kaiser → {n_kaiser} PCs')
axes[1].set_title('Cumulative Variance'); axes[1].set_xlabel('PCs')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()
print(f'Kaiser: {n_kaiser} | 90%: {n_90} | 95%: {n_95}')

In [ ]:
def compute_rmsecv(X_s, max_k, G=10, rs=42):
    n, p  = X_s.shape; max_k = min(max_k, p, n-1)
    idx   = np.arange(n); sg = [idx[g::G] for g in range(G)]
    vg    = [np.array([j]) for j in range(p)]
    press = np.zeros(max_k); rmsecv = np.zeros(max_k)
    for k in range(1, max_k+1):
        PRESS=0; COUNT=0
        for ti in sg:
            tri = np.setdiff1d(idx, ti); Xtr=X_s[tri]; Xte=X_s[ti]
            pc  = PCA(n_components=min(k,p), svd_solver='full', random_state=rs)
            pc.fit(Xtr); P=pc.components_.T
            for mc in vg:
                oc=np.setdiff1d(np.arange(p),mc); kk=min(k,P[oc,:].shape[0]-1)
                if kk<1: continue
                Th,*_=np.linalg.lstsq(P[oc,:kk],Xte[:,oc].T,rcond=None)
                Xh=(Th.T)@P[:,:kk].T; r=Xte[:,mc]-Xh[:,mc]
                PRESS+=np.sum(r**2); COUNT+=r.size
        press[k-1]=PRESS; rmsecv[k-1]=np.sqrt(PRESS/COUNT) if COUNT>0 else np.inf
    return int(np.argmin(rmsecv))+1, press, rmsecv

print('Running RMSECV cross-validation...')
best_k_cv, press_cv, rmsecv_cv = compute_rmsecv(X_train_s_eval, max_k, CV_FOLDS, RANDOM_STATE)
ks = np.arange(1, len(rmsecv_cv)+1)
fig, axes = plt.subplots(1, 2, figsize=(14,4))
for ax, vals, yl, tit in [(axes[0],press_cv,'PRESS','PRESS'),(axes[1],rmsecv_cv,'RMSECV','RMSECV')]:
    ax.plot(ks, vals, marker='o')
    ax.axvline(best_k_cv, color='red', linestyle='--', label=f'min @ k={best_k_cv}')
    ax.set_title(tit); ax.set_xlabel('PCs'); ax.set_ylabel(yl)
    ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()
print(f'✅ RMSECV best k = {best_k_cv}')

In [ ]:
k_final = N_COMPONENTS if N_COMPONENTS is not None else best_k_cv
print('Component selection summary')
print(f'  Kaiser   : {n_kaiser} PCs')
print(f'  90% var  : {n_90} PCs')
print(f'  95% var  : {n_95} PCs')
print(f'  RMSECV   : {best_k_cv} PCs')
print(f'  Selected : {k_final} PCs  ({"user" if N_COMPONENTS else "RMSECV"})')
summary_df = pd.DataFrame({
    'Eigenvalue': eigenvalues[:max_k].round(4),
    'Variance_%': (var_ratio[:max_k]*100).round(3),
    'Cum_Var_%':  (cum_var[:max_k]*100).round(3),
    'PRESS':      press_cv[:max_k].round(4),
    'RMSECV':     rmsecv_cv[:max_k].round(6),
}, index=[f'PC{i}' for i in range(1, max_k+1)])
summary_df.index.name='Component'
display(summary_df)

---
# SECTION 8 — PCA-SPC MODEL FIT (Phase I)

In [ ]:
def fit_pca_spc(df_tr, k, alpha, feature_names, rs=42):
    X = df_tr[feature_names].to_numpy(); n,p = X.shape
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    pca = PCA(n_components=k, svd_solver='full', random_state=rs)
    T   = pca.fit_transform(Xs); P=pca.components_.T; lam=pca.explained_variance_
    T2  = np.sum((T**2)/lam, axis=1)
    T2_UCL = (k*(n-1)/(n-k))*f.ppf(alpha,k,n-k)
    E   = Xs - pca.inverse_transform(T); Q=np.sum(E**2,axis=1)
    pf  = PCA(n_components=min(p,n-1),svd_solver='full',random_state=rs); pf.fit(Xs)
    re  = pf.explained_variance_[k:]
    t1,t2,t3=re.sum(),(re**2).sum(),(re**3).sum()
    h0  = 1-(2*t1*t3)/(3*t2**2); z=norm.ppf(alpha)
    Q_UCL = t1*((z*np.sqrt(2*t2*h0**2)/t1)+1+(t2*h0*(h0-1)/t1**2))**(1/h0)
    def transform_new(df_new):
        Xn=df_new[feature_names].to_numpy(); Xns=sc.transform(Xn)
        Tn=pca.transform(Xns); T2n=np.sum((Tn**2)/lam,axis=1)
        En=Xns-pca.inverse_transform(Tn); Qn=np.sum(En**2,axis=1)
        out=pd.DataFrame(Tn,columns=[f'PC{i}' for i in range(1,k+1)],index=df_new.index)
        out['T2']=T2n; out['Q']=Qn
        out['T2_flag']=T2n>T2_UCL; out['Q_flag']=Qn>Q_UCL
        return out
    return {'feature_names':feature_names,'scaler':sc,'pca':pca,'k':k,
            'scores':T,'loadings':P,'eigenvalues_k':lam,
            'T2':T2,'Q':Q,'T2_UCL':T2_UCL,'Q_UCL':Q_UCL,
            'X_scaled_train':Xs,'E_train':E,'transform_new':transform_new}

results = fit_pca_spc(df_train_clean, k_final, ALPHA_CONTROL, FEATURE_NAMES, RANDOM_STATE)
print(f'✅ Model fitted | k={k_final} | T² UCL={results["T2_UCL"]:.4f} | Q UCL={results["Q_UCL"]:.4f}')

---
# SECTION 9 — PHASE I & II CHARTS

In [ ]:
T=results['scores']; P=results['loadings']; k=results['k']
lam=results['eigenvalues_k']; T2=results['T2']; Q=results['Q']
T2_UCL=results['T2_UCL']; Q_UCL=results['Q_UCL']
names=results['feature_names']; pca=results['pca']
evr=pca.explained_variance_ratio_*100
n=T.shape[0]; p=P.shape[0]
flagged=((T2>T2_UCL)|(Q>Q_UCL))

# Score plot PC1 vs PC2
ang=np.linspace(0,2*np.pi,400)
a=np.sqrt(T2_UCL*lam[0]); b=np.sqrt(T2_UCL*lam[1])
plt.figure(figsize=(7,6))
plt.scatter(T[:,0],T[:,1],s=15,alpha=0.6,label='In-control')
plt.scatter(T[flagged,0],T[flagged,1],s=40,marker='x',color='red',label='Flagged')
plt.plot(a*np.cos(ang),b*np.sin(ang),'k--',lw=1.5,label='T² UCL')
plt.axhline(0,color='grey',lw=0.8); plt.axvline(0,color='grey',lw=0.8)
plt.xlabel(f'PC1 ({evr[0]:.2f}%)'); plt.ylabel(f'PC2 ({evr[1]:.2f}%)')
plt.title('PHASE I — Score plot PC1 vs PC2'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

for vals, ucl, title, color in [
    (T2, T2_UCL, 'PHASE I — Hotelling T²', 'steelblue'),
    (Q,  Q_UCL,  'PHASE I — Q (SPE)',       'darkorange')]:
    plt.figure(figsize=(12,4))
    plt.plot(vals, marker='o', lw=1, ms=3, color=color)
    plt.axhline(ucl, color='red', linestyle='--', lw=1.5, label=f'UCL={ucl:.2f}')
    plt.title(title); plt.xlabel('Observation'); plt.grid(True)
    plt.legend(); plt.tight_layout(); plt.show()

# Loadings heatmap
plt.figure(figsize=(min(k*1.2+2,16), max(5,0.3*p)))
sns.heatmap(P, cmap='coolwarm', center=0, annot=(p<=30), fmt='.2f', linewidths=0.3,
            yticklabels=names,
            xticklabels=[f'PC{i+1}\n({evr[i]:.1f}%)' for i in range(k)])
plt.title('Loadings Heatmap'); plt.tight_layout(); plt.show()
print(f'Phase I residual flags: {flagged.sum()}/{n} (should be ~0 after cleaning)')

# Phase II
test_monitor = results['transform_new'](df_test_X)
T2t=test_monitor['T2'].values; Qt=test_monitor['Q'].values
fT2t=test_monitor['T2_flag'].values; fQt=test_monitor['Q_flag'].values
nt=len(T2t)

for vals, flags, ucl, title, color in [
    (T2t, fT2t, T2_UCL, 'PHASE II — Hotelling T²', 'steelblue'),
    (Qt,  fQt,  Q_UCL,  'PHASE II — Q (SPE)',       'darkorange')]:
    plt.figure(figsize=(12,4))
    plt.plot(vals, marker='o', lw=1, ms=3, color=color)
    plt.axhline(ucl, color='red', linestyle='--', lw=1.5, label=f'UCL={ucl:.2f}')
    for r in np.where(flags)[0]: plt.text(r, vals[r], str(r), fontsize=7, color='red')
    plt.title(title); plt.xlabel('Test Observation'); plt.grid(True)
    plt.legend(); plt.tight_layout(); plt.show()

print(f'Phase II | T² flagged: {fT2t.sum()}/{nt} ({fT2t.mean()*100:.1f}%) | Q flagged: {fQt.sum()}/{nt} ({fQt.mean()*100:.1f}%)')

---
# SECTION 10 — CONTRIBUTION PLOTS (Root Cause)

In [ ]:
def compute_contrib_limits(res, df_tr, alpha_sig=0.05):
    nm=res['feature_names']; sc=res['scaler']; pc=res['pca']; P=res['loadings']; lam=res['eigenvalues_k']
    Xs=sc.transform(df_tr[nm].to_numpy()); T=pc.transform(Xs); E=Xs-pc.inverse_transform(T)
    z=norm.ppf(1-alpha_sig/2)
    mu_e=E.mean(0); sd_e=E.std(0,ddof=1)
    res['Qcontrib_LCL']=mu_e-z*sd_e; res['Qcontrib_UCL']=mu_e+z*sd_e
    W=T/lam; cT2=Xs*(W@P.T); mu_c=cT2.mean(0); sd_c=cT2.std(0,ddof=1)
    res['T2contrib_LCL']=mu_c-z*sd_c; res['T2contrib_UCL']=mu_c+z*sd_c
    return res

def plot_top_contrib(res, df_te, top_n=5, use_names=False):
    nm=res['feature_names']; sc=res['scaler']; pc=res['pca']
    P=res['loadings']; lam=res['eigenvalues_k']
    T2_UCL=res['T2_UCL']; Q_UCL=res['Q_UCL']
    Xns=sc.transform(df_te[nm].to_numpy()); Tn=pc.transform(Xns)
    T2n=np.sum((Tn**2)/lam,axis=1); En=Xns-pc.inverse_transform(Tn); Qn=np.sum(En**2,axis=1)
    xax=np.arange(1,len(nm)+1); xlbl=nm if use_names else xax
    for stat,vals,ucl,lk,uk,label,col in [
        ('Q',Qn,Q_UCL,'Qcontrib_LCL','Qcontrib_UCL','Q (SPE)','darkorange'),
        ('T2',T2n,T2_UCL,'T2contrib_LCL','T2contrib_UCL','T²','steelblue')]:
        fi=np.where(vals>ucl)[0]; si=fi[np.argsort(vals[fi])[::-1]][:top_n]
        print(f'{label} flagged: {len(fi)} | showing top {len(si)}')
        for r in si:
            c=En[r,:] if stat=='Q' else Xns[r,:]*(P@(Tn[r,:]/lam))
            plt.figure(figsize=(12,4))
            plt.bar(xax,c,color=col,alpha=0.7)
            plt.plot(xax,res[uk],'r-',lw=1.5,label='UCL')
            plt.plot(xax,res[lk],'r-',lw=1.5,label='LCL')
            plt.axhline(0,color='black',lw=0.8)
            plt.title(f'{label} Contribution — Obs {df_te.index[r]} | {stat}={vals[r]:.3f} | {vals[r]/ucl:.2f}×UCL')
            plt.xticks(xax,xlbl,rotation=90 if use_names else 0,fontsize=7)
            plt.legend(); plt.grid(True,axis='y'); plt.tight_layout(); plt.show()

results = compute_contrib_limits(results, df_train_clean)
plot_top_contrib(results, df_test_X, top_n=TOP_N_CONTRIB, use_names=USE_VAR_NAMES_ON_X)

---
# SECTION 11 — Y VALIDATION

> Runs only if `Y_COLS` is defined in Section 1.
>
> Checks whether flagged cycles also have quality variables (Y) outside tolerance.
> - **High overlap** → model detects real quality issues ✅
> - **Low overlap** → model detects process variation before it affects quality (early warning)
> - **No overlap** → model may need tuning
>
> Note: the press already signals Y out-of-tolerance in real time.
> PCA-SPC value = **early warning** + **root cause diagnosis**.

In [ ]:
if df_test_Y.empty:
    print('No Y variables defined. Set Y_COLS in Section 1 to enable Y validation.')
else:
    # Build tolerance limits
    tol = {}
    if TOLERANCE_SPECS:
        for col, spec in TOLERANCE_SPECS.items():
            if col not in df_train_Y_clean.columns: continue
            mu = df_train_Y_clean[col].mean()
            if spec[0]=='absolute': tol[col]={'mean':mu,'lower':mu-spec[1],'upper':mu+spec[1]}
            elif spec[0]=='relative': tol[col]={'mean':mu,'lower':mu*(1-spec[1]),'upper':mu*(1+spec[1])}
            elif spec[0]=='fixed': tol[col]={'mean':mu,'lower':spec[1],'upper':spec[2]}
    else:
        for col in df_test_Y.columns:
            if col not in (df_train_Y_clean.columns if not df_train_Y_clean.empty else []): continue
            mu=df_train_Y_clean[col].mean(); sig=df_train_Y_clean[col].std()
            tol[col]={'mean':mu,'lower':mu-2*sig,'upper':mu+2*sig}

    ooc_mask   = test_monitor['T2_flag']|test_monitor['Q_flag']
    flagged_idx = test_monitor[ooc_mask].index
    rows=[]
    for idx in flagged_idx:
        row={'Cycle':idx,'T2':round(test_monitor.loc[idx,'T2'],4),
             'Q':round(test_monitor.loc[idx,'Q'],4),
             'T2_flag':test_monitor.loc[idx,'T2_flag'],
             'Q_flag':test_monitor.loc[idx,'Q_flag']}
        for col,lims in tol.items():
            val=float(df_test_Y.loc[idx,col])
            row[f'{col}_val']=round(val,4)
            row[f'{col}_lower']=round(lims['lower'],4)
            row[f'{col}_upper']=round(lims['upper'],4)
            row[f'{col}_OOT']=not(lims['lower']<=val<=lims['upper'])
        rows.append(row)

    report_df = pd.DataFrame(rows).set_index('Cycle')
    oot_cols  = [c for c in report_df.columns if c.endswith('_OOT')]
    n_flagged = len(flagged_idx)

    print(f'Phase II: {len(df_test_X)} test cycles | {n_flagged} flagged by PCA-SPC ({n_flagged/len(df_test_X)*100:.1f}%)')

    if oot_cols:
        oot_counts = report_df[oot_cols].sum().rename(lambda x: x.replace('_OOT',''))
        oot_counts = oot_counts[oot_counts>0].sort_values(ascending=False)
        if len(oot_counts)>0:
            print('\nOut-of-tolerance Y variables (among flagged cycles):')
            for var,cnt in oot_counts.items():
                print(f'  {var:40s}: {cnt} cycles ({cnt/n_flagged*100:.1f}% of flagged)')
            plt.figure(figsize=(min(12,len(oot_counts)+2),4))
            oot_counts.plot(kind='bar',color='tomato',edgecolor='black')
            plt.title('Out-of-Tolerance per Y Variable (flagged cycles)')
            plt.ylabel('Count'); plt.xticks(rotation=45,ha='right')
            plt.grid(True,axis='y'); plt.tight_layout(); plt.show()
        else:
            print('\nNo Y out of tolerance in flagged cycles.')
            print('→ PCA-SPC may be detecting early process drift not yet visible in quality.')
    print()
    display(report_df.head(20))

---
# SECTION 12 — FINAL SUMMARY

In [ ]:
print('='*55)
print('  PCA-SPC ANALYSIS — FINAL SUMMARY')
print('='*55)
print(f'  Dataset          : {DATA_SOURCE.upper()}')
print(f'  Total obs        : {len(df_X)}')
print(f'  X variables      : {len(FEATURE_NAMES)}')
print(f'  Y variables      : {len(Y_COLS_valid)} → {Y_COLS_valid}')
print(f'  Split type       : {SPLIT_TYPE}')
print(f'  Phase I (clean)  : {len(df_train_clean)} obs'
      + (f' ({removed_total} removed)' if PHASE1_CLEANING and removed_total>0 else ''))
print(f'  Phase II (test)  : {len(df_test_X)} obs')
print(f'  PCs selected     : {k_final}')
print(f'  T² UCL (α={ALPHA_CONTROL}): {results["T2_UCL"]:.4f}')
print(f'  Q  UCL (α={ALPHA_CONTROL}): {results["Q_UCL"]:.4f}')
print(f'  T² flagged (II)  : {fT2t.sum()} ({fT2t.mean()*100:.1f}%)')
print(f'  Q  flagged (II)  : {fQt.sum()}  ({fQt.mean()*100:.1f}%)')
print('='*55)
print()
print('Next steps:')
print('  1. Interpret contribution plots → identify root cause variables')
if Y_COLS_valid:
    print('  2. Y validation → do flagged cycles match quality issues?')
    print('  3. If yes → model ready for early warning deployment')
    print('  4. Future: build PLS model to predict Y from X in real time')
else:
    print('  2. Add Y_COLS in Section 1 to enable quality validation')
    print('  3. Future: build PLS model (quality prediction from process variables)')